In [1]:
import pandas as pd
import re
from thefuzz import process

print("Loading datasets with structural constraints...")
df_main = pd.read_csv("lagos_rent_cleaned_locations.csv")
target_col = 'Location' if 'Location' in df_main.columns else df_main.columns[1]

# USING YOUR NEW CLEANED DICTIONARY
df_dict = pd.read_csv("Official_Lagos_Dictionary_Cleaned.csv")
df_dict_unique = df_dict.drop_duplicates(subset=['Sub_Location'], keep='first')

# Sort official locations longest-to-shortest
official_sub_locs = df_dict_unique['Sub_Location'].astype(str).tolist()
official_sub_locs.sort(key=len, reverse=True)

unique_messy_locs = df_main[target_col].dropna().unique()
mapping_dict = {}

print("Executing Length-Sorted Proximity Mapper...")

def find_smart_match(address, sorted_official_list):
    address_clean = str(address).strip().title()
    
    # EXACT MATCHING
    exact_matches = []
    for loc in sorted_official_list:
        if re.search(r'\b' + re.escape(loc) + r'\b', address_clean, re.IGNORECASE):
            exact_matches.append(loc)
            
    if exact_matches:
        longest_matches = []
        for match in exact_matches:
            if not any(match != other and match.lower() in other.lower() for other in exact_matches):
                longest_matches.append(match)
                
        if len(longest_matches) == 1:
            return longest_matches[0]
        
        # PROXIMITY ANCHORING
        if len(longest_matches) > 1:
            best_match = None
            min_distance_from_end = float('inf')
            for match in longest_matches:
                match_index = address_clean.lower().rfind(match.lower())
                distance = len(address_clean) - match_index
                if distance < min_distance_from_end:
                    min_distance_from_end = distance
                    best_match = match
            return best_match

    # FUZZY MATCHING (Fallback)
    tail_cleaned = re.sub(r'\b(lagos state|lagos|nigeria)\b', '', address_clean, flags=re.IGNORECASE).strip()
    segments = [seg.strip() for seg in tail_cleaned.split(',') if seg.strip()]
    if segments:
        target_zone = " ".join(segments[-2:])
        match = process.extractOne(target_zone, sorted_official_list, score_cutoff=80)
        if match:
            return match[0]

    match = process.extractOne(address_clean, sorted_official_list, score_cutoff=80)
    return match[0] if match else "Unmapped"

# Run mapping
for messy in unique_messy_locs:
    mapping_dict[str(messy)] = find_smart_match(messy, official_sub_locs)

print("Writing clean CSV output...")
df_main['Official_Sub_Location'] = df_main[target_col].astype(str).map(mapping_dict)

df_final = df_main.merge(
    df_dict_unique[['Sub_Location', 'Main_District']], 
    how='left', 
    left_on='Official_Sub_Location', 
    right_on='Sub_Location'
)

if 'Sub_Location' in df_final.columns:
    df_final.drop(columns=['Sub_Location'], inplace=True)

df_final.to_csv("lagos_rent_mapped_final.csv", index=False)
print("SUCCESS: Master geography is mapped and saved as 'lagos_rent_mapped_final.csv'!")

Loading datasets with structural constraints...
Executing Length-Sorted Proximity Mapper...
Writing clean CSV output...
SUCCESS: Master geography is mapped and saved as 'lagos_rent_mapped_final.csv'!


In [2]:
import pandas as pd

print("Generating readable text trees...")

# Load the updated datasets
df_dict = pd.read_csv("Official_Lagos_Dictionary_Cleaned.csv")
df_main = pd.read_csv("lagos_rent_mapped_final.csv")
target_col = 'Location' if 'Location' in df_main.columns else df_main.columns[1]

# ---------------------------------------------------------
# FILE 1: The Clean Dictionary Tree
# ---------------------------------------------------------
with open("Lagos_Dictionary_Tree.txt", "w", encoding="utf-8") as f:
    f.write("=== OFFICIAL LAGOS LOCATION DICTIONARY ===\n\n")
    
    grouped_dict = df_dict.groupby('Main_District')
    for district, group in sorted(grouped_dict):
        f.write(f"📍 {district}\n")
        sub_locs = sorted(group['Sub_Location'].astype(str).unique())
        for sub in sub_locs:
            f.write(f"    ├── {sub}\n")
        f.write("\n")

print("-> Created 'Lagos_Dictionary_Tree.txt'")

# ---------------------------------------------------------
# FILE 2: The Mapping Traceability Tree
# ---------------------------------------------------------
with open("Lagos_Mapped_Strings_Tree.txt", "w", encoding="utf-8") as f:
    f.write("=== HOW MESSY STRINGS MAPPED TO OFFICIAL LOCATIONS ===\n\n")
    
    df_mapped = df_main[df_main['Official_Sub_Location'] != 'Unmapped']
    grouped_main = df_mapped.groupby(['Main_District', 'Official_Sub_Location'])
    
    current_district = ""
    for (district, sub_loc), group in sorted(grouped_main):
        if district != current_district:
            f.write(f"\n📍 {district}\n")
            f.write(f"{'='*len(str(district)) + '===='}\n")
            current_district = district
        
        f.write(f"  📂 {sub_loc}\n")
        
        messy_strings = sorted(group[target_col].astype(str).unique())
        for messy in messy_strings:
            f.write(f"      ├── {messy}\n")
        f.write("\n")

print("-> Created 'Lagos_Mapped_Strings_Tree.txt'")
print("\nSUCCESS: Both files are ready to view in your project folder!")

Generating readable text trees...
-> Created 'Lagos_Dictionary_Tree.txt'
-> Created 'Lagos_Mapped_Strings_Tree.txt'

SUCCESS: Both files are ready to view in your project folder!
